# testing BB_wrapper

In [1]:
import BB_wrapper
import torch

/Users/karim/projects/k-sPSS/.venv/lib/python3.11/site-packages/pycutest/__init__.py:26: RuntimeWarning: the PYCUTEST_CACHE environment variable is not set; current folder will be used for caching.
  warnings.warn("the PYCUTEST_CACHE environment variable is not set; current folder will be used for caching.", RuntimeWarning)


testing cutest collection wrapper

In [2]:
cutest_wrapper = BB_wrapper.BB_cutest_collection(write_to_file="cutest_problem_selection.txt", cap_n_problems=2)

loading problems...
0/276
loaded 2 problems


In [3]:
# evaluating n problem functions
for i in range(2):
    p = cutest_wrapper.problems[i]
    f = cutest_wrapper.problem_functions[i]
    print(f"{p.name} | n: {p.n}: {f(torch.from_numpy(p.x0))}")

SISSER | n: 2: 3.02030030003
CHWIRUT2LS | n: 3: 14794.790154797307


k-fail wrapper

In [164]:
problem_idx = 1
k = 50
n_dim = 3
k_fail_wrapper = BB_wrapper.BB_k_fail_wrapper(cutest_wrapper.problem_functions[problem_idx], k*torch.ones((1024, 2), dtype=torch.int16))

In [165]:
k_fail_wrapper.batch_call(torch.rand((k + 2, n_dim)))

(tensor([87206.7109, 88795.8281]), tensor([38, 23], dtype=torch.int32))

# Testing with ga


In [166]:
# custom function for batch processing
def f_batch(x_batch):
    vals, idxs = k_fail_wrapper.batch_call(x_batch)
    print(vals, idxs)
    # for those who failed set function to +1e9
    out = torch.ones(x_batch.shape[0]) * 1e9
    print(out)
    out[idxs] = vals
    return out

In [167]:
def function_valued_fitness(p, f, c=1): # MODIFIED FOR BATCH PROCESSING
    # evaluate population
    xs = torch.stack([x for x in p])
    f_vals = f_batch(xs)
    f_bar = torch.max(f_vals)
    return -f_vals + f_bar + c
    

def is_select_parents(p_fit=None):
    return torch.rand(1) <= 0.8 # TODO

def elitism_selection(p, p_fit, n, p_next=[]):
    p_fit_copy = p_fit.clone()
    for i, x in enumerate(p_next):
        if (x == p).all(dim=1).any().item(): # is x in p
            p_fit_copy[i] = 0
    
    _, selexted_idx = torch.topk(p_fit_copy, n)
    return p[selexted_idx]

def weighted_average_crossover(parents_tensor, get_weights=None, p_fit=None):
    weights_tensor = torch.eye(parents_tensor.shape[1]) / parents_tensor.shape[0]
    if not (get_weights is None or p_fit is None):
        weights_tensor = get_weights(p_fit)

    # parents and weights must be tensors
    return torch.sum(parents_tensor @ weights_tensor, dim=0)

def breed_parents_default(p, p_fit, selection, crossover):
    # select two parents
    parent_tensor = selection(p, p_fit, 2)

    # crossover
    return crossover(parent_tensor) # TODO: add weight function

def mutate_normal_distr(x, c=1e-1):
    return x + c * torch.randn_like(x)

class point_reuse:
        def __init__(self, f):
            self.f = f
            self.f_points = {}
            self.points_raw = []
        
        def evaluate(self, x):
            x_hash = x.numpy().tobytes()

            if x_hash in self.f_points.keys(): # already evaluated at this exact point
                return self.f_points[x_hash]
            else: # must evaluate from scratch
                val = self.f(x)
                self.f_points[x_hash] = val
                self.points_raw.append(x)
                return val # 1 stands for 1 evaluation of the function
        
        def get_n_f_evals(self):
            return len(self.f_points)
        
        def get_evals(self): # returns dict of all evaluations and corresponding values
            points = [[], []]
            
            for x in self.points_raw:
                x_hash = x.numpy().tobytes()
                points[0].append(x)
                points[1].append(self.f_points[x_hash])
            
            return points


class GA:
    def __init__(self, p, f):
        self.p = p
        self.f = f
        # 0 - init
        self.k = 0
        self.p_bar = p.shape[0]
        self.p_reuse = point_reuse(f)

    def step(self, mutation_prob=0, fit=function_valued_fitness, is_select_parents=is_select_parents, selection=elitism_selection, crossover=weighted_average_crossover, breed_parents = breed_parents_default, mutate=lambda x: x, is_in_domain=lambda x: True):
        # 1 - fitness (also evaluates function points)
        p_fit = fit(self.p, self.p_reuse.evaluate) # returns a tensor of p_bar elements

        # 2 - reproduce (saturate next population)
        p_next = []
        p_next_bar = 0
        while p_next_bar < self.p_bar: # 2d saturation check
            if is_select_parents(): # selecting survivor(s)
                # 2a - select parents and 2b - perform crossover
                offspring = breed_parents(self.p, p_fit, selection, crossover)
                
                # 2c mutation
                if torch.rand(1) <= mutation_prob:
                    offspring = mutate(offspring)

                # 2d - update next generation (if not deceased)
                if is_in_domain(offspring) and not any(torch.equal(offspring, t) for t in p_next):
                    p_next.append(offspring)
                    p_next_bar += 1
            else: # selecting survivor
                survivor = selection(self.p, p_fit, 1, p_next=p_next)[0] # note! selection return a tensor of tensors
                
                if is_in_domain(survivor) and not any(torch.equal(survivor, t) for t in p_next):
                    p_next.append(survivor)
                    p_next_bar += 1
        
        # set p_next as current generation
        self.p = torch.stack(p_next)
    
    def get_population_best(self): # return best f value in current population
        min_f = k_fail_wrapper.f(self.p[0])
        for i in range(1, self.p_bar):
            min_f = min(min_f, k_fail_wrapper.f(self.p[i]))
        return min_f



In [168]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

def GA_evaluation(alg, mutation_prob=0, mutate=mutate_normal_distr, is_in_domain=lambda x: True, selection=elitism_selection, function_call_cap=100, k_cap=100, display=False, title=""):
    def display_progress():
        clear_output(wait=True)
        plt.plot(pop_best_point)
        plt.title(title + " | " + str(alg.p_reuse.get_n_f_evals()))
        plt.show()
    
    pop_best_point = []
    pop_best_point.append(alg.get_population_best())

    k = 0
    while alg.p_reuse.get_n_f_evals() < function_call_cap and k < k_cap:
        alg.step(mutation_prob=mutation_prob, mutate=mutate, selection=selection, is_in_domain=is_in_domain)

        pop_best_point.append(alg.get_population_best())

        k += 1
    
        if display:
            display_progress()
    
    return pop_best_point

In [169]:
from functools import partial

p0 = torch.rand((100, n_dim)) * 10 + 0*5 # OFFSET AND STILL WORKS FOR PROBLEM 0, FOR PROBLEM 1 REMOVE OFFSET AND INCREASE POP 10->50 or 100
alg_1 = GA(p0, cutest_wrapper.problem_functions[problem_idx]) # function will not be called as we have overriden fit
alg_1_perf = GA_evaluation(alg_1, mutation_prob=1.0, mutate=partial(mutate_normal_distr, c=1e-2), display=True, title="main", function_call_cap=10, k_cap=600)

KeyboardInterrupt: 

In [102]:
print(alg_1.get_population_best())

for x in alg_1.p:
    print(x, k_fail_wrapper.f(x))

40316.880710335376
tensor([-0.9115, -0.5166,  1.1440]) 60755.55713275221
tensor([-0.8780, -0.5322,  1.2100]) 64623.30823558253
tensor([-0.8554, -0.5241,  1.2740]) 70861.81031899007
tensor([-0.9592, -0.6380,  1.1750]) 122319.72616831175
tensor([-0.9808, -0.6294,  1.1230]) 114260.8067415483
tensor([-0.8829, -0.5906,  1.2191]) 41356.20086812114
tensor([-0.9279, -0.5806,  1.2143]) 46547.40741139544
tensor([-0.8517, -0.5403,  1.1178]) 40316.880710335376
tensor([-0.8801, -0.5851,  1.1069]) 142981.13998737247
tensor([-0.8099, -0.5850,  1.1774]) 642677.511283199


In [ ]:
print(cutest_wrapper.problem_functions[problem_idx](torch.tensor([-0.0819, -0.3601])))
print(cutest_wrapper.problem_functions[problem_idx](torch.tensor([0, 0])))

0.052319050881209084
0.0


In [ ]:
plt.plot(alg_1_perf, label="1")
plt.plot(alg_2_perf, label="2")
plt.plot(alg_3_perf, label="3")
plt.legend()
plt.show()

In [ ]:
plt.plot(alg_1.p_reuse.get_evals()[1], label="1")
plt.plot(alg_2.p_reuse.get_evals()[1], label="2")
plt.plot(alg_3.p_reuse.get_evals()[1], label="3")
plt.legend()
plt.show()

In [10]:
a = torch.zeros(10, dtype=int)
a

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [11]:
a[torch.tensor([0, 2, 6, 1])] = torch.arange(1, 10, 1)[torch.tensor([0, 2, 6, 1])]

In [12]:
a

tensor([1, 2, 3, 0, 0, 0, 7, 0, 0, 0])